# 📖 Routers and Skills

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. **Understand** what routing is in agents
2. **Implement** skill-based routing
3. **Build** multi-skill agent systems
4. **Apply** routing for different task types

---

## ⏱️ Time Estimate

**~30 minutes**

## 📦 Setup

In [ ]:
!pip install -q openai langchain langchain-core
import os
if "OPENAI_API_KEY" not in os.environ:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")

---

## 🧠 Theory: What is Routing?

**Routing** is the agent's ability to **direct** requests to the appropriate handler based on what the user is asking for.

### Real-World Analogy

```
┌─────────────────────────────────────────────────────┐
│          CUSTOMER SERVICE ANALOGY                  │
├─────────────────────────────────────────────────────┤
│                                             │
│  Customer: "I want to return a product"         │
│  Router: → Transfers to RETURNS department   │
│                                             │
│  Customer: "My bill is wrong"                │
│  Router: → Transfers to BILLING department     │
│                                             │
│  Customer: "Where are you located?"           │
│  Router: → Can handle directly, no transfer   │
│                                             │
└─────────────────────────────────────────────────────┘
```

### Types of Routing

| Type | Description | Use Case |
|-----|-------------|----------|
**| **Intent-based** | Classify user intent | Customer support |
**| **Skill-based** | Route to capability | Multi-tool agents |
**| **Conditional** | If/else logic | Workflows |
**| **LLM-based** | Let LLM decide | Dynamic routing |

---

## 💻 Code Along: Building a Router

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Define specialized tools/skills
@tool
def search_web(query: str) -> str:
    """Search the web for information.
    
    Args:
        query: Search query
    
    Returns:
        Search results
    """
    return f"[Search results for '{query}'] Found 10 articles."

@tool
def calculate(expression: str) -> str:
    """Calculate a mathematical expression.
    
    Args:
        expression: Math expression like '2+2' or 'sqrt(16)'
    
    Returns:
        Result
    """
    return str(eval(expression))

@tool
def get_weather(location: str) -> str:
    """Get weather for a location.
    
    Args:
        location: City name
    
    Returns:
        Weather info
    """
    return f"Weather in {location}: Sunny, 72°F"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email.
    
    Args:
        to: Email recipient
        subject: Email subject
        body: Email body
    
    Returns:
        Confirmation
    """
    return f"Email sent to {to}"

# All skills
all_tools = [search_web, calculate, get_weather, send_email]
print("✅ Tools/Skills defined:")
for t in all_tools:
    print(f"   • {t.name}")

### Method 1: LLM-Based Routing (Let the Model Decide)

In [ ]:
# Method 1: Let the LLM route by tool selection
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(all_tools)

test_queries = [
    "What's 15 * 23?",
    "What's the weather in Tokyo?",
    "Search for Python tutorials",
    "Send an email to john@example.com about the meeting"
]

print("📍 LLM-Based Routing Demo")
print("=" * 60)

for query in test_queries:
    response = llm_with_tools.invoke(query)
    tool_called = response.tool_calls[0]['name'] if response.tool_calls else "none"
    print(f"Query: '{query}'")
    print(f"  → Routed to: {tool_called}")
    print()

### Method 2: Explicit Intent Classification

Let's build a more controlled router:

In [ ]:
# Simple intent-based router
def classify_intent(query: str) -> str:
    """Classify the user intent."""
    query_lower = query.lower()
    
    # Simple rule-based classification
    if any(w in query_lower for w in ['calculate', 'math', '+', '-', '*', '/']):
        return "calculator"
    elif any(w in query_lower for w in ['weather', 'temperature', 'forecast']):
        return "weather"
    elif any(w in query_lower for w in ['search', 'find', 'look up']):
        return "search"
    elif any(w in query_lower for w in ['email', 'send', 'mail']):
        return "email"
    else:
        return "general"

# Test it
print("📍 Intent Classification Routing")
print("=" * 60)

for query in test_queries:
    intent = classify_intent(query)
    print(f"Query: '{query}'")
    print(f"  → Intent: {intent}")
    print()

### Method 3: LLM-Based Intent Classification

More powerful - let the LLM classify:

In [ ]:
# Use LLM to classify intent
llm_router = ChatOpenAI(model="gpt-4o-mini")

intent_system_prompt = """You are a intent classifier. Classify user queries into one of these intents:
- calculator: Math problems, calculations
- weather: Weather-related questions
- search: Looking for information
- email: Sending emails
- general: Anything else

Respond ONLY with the intent name."""

def llm_route(query: str) -> str:
    """Route using LLM."""
    response = llm_router.invoke([
        SystemMessage(content=intent_system_prompt),
        HumanMessage(content=query)
    ])
    return response.content.strip().lower()

print("📍 LLM-Based Intent Routing")
print("=" * 60)

for query in test_queries:
    intent = llm_route(query)
    print(f"Query: '{query}'")
    print(f"  → Intent: {intent}")
    print()

---

## 🔧 Multi-Skill Agent Architecture

Let's build a complete multi-skill agent:

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# Define skill handlers
def handle_calculator(query: str) -> str:
    """Handle math queries."""
    llm = ChatOpenAI(model="gpt-4o-mini")
    return llm.invoke(f"Calculate: {query}").content

def handle_weather(query: str) -> str:
    """Handle weather queries."""
    # Extract location (simplified)
    return f"Weather: 72°F Sunny (location extracted from query)"

def handle_search(query: str) -> str:
    """Handle search queries."""
    return f"[Search results for '{query}']"

def handle_email(query: str) -> str:
    """Handle email queries."""
    return "Email composed and ready to send"

def handle_general(query: str) -> str:
    """Handle general queries."""
    llm = ChatOpenAI(model="gpt-4o-mini")
    return llm.invoke(query).content

# Create router
skill_handlers = {
    "calculator": handle_calculator,
    "weather": handle_weather,
    "search": handle_search,
    "email": handle_email,
    "general": handle_general
}

def multi_skill_agent(query: str) -> str:
    """Route and handle with appropriate skill."""
    # Classify intent
    intent = llm_route(query)
    print(f"📍 Routing to: {intent}")
    
    # Get handler
    handler = skill_handlers.get(intent, handle_general)
    
    # Execute
    result = handler(query)
    return result

# Test
result = multi_skill_agent("What's the weather in Paris?")
print(f"Result: {result}")

### Multi-Skill Architecture Diagram

```
┌─────────────────────────────────────────────────────────┐
│              MULTI-SKILL AGENT                          │
├───────────────────────────────────���─���───────────────────┤
│                                                 │
│  User Query                                        │
│     │                                             │
│     ▼                                             │
│  ┌─────────────┐                                   │
│  │  ROUTER    │ ← Intent Classification           │
│  └──────┬──────┘                                   │
│         │                                          │
│    ┌────┴────┬────────┬────────┐                   │
│    ▼        ▼        ▼        ▼                   │
│ Calculator Weather  Search   Email               │
│    │        │        │        │                   │
│    └────────┴────────┴────────┘                   │
│              │                                    │
│              ▼                                    │
│         Result                                     │
└─────────────────────────────────────────────────────────┘
```

---

## 🧪 Try It Yourself!

**Exercise 1**: Add a new skill for code generation.
**Exercise 2**: Make the router handle multiple intents.
**Exercise 3**: Add fallthrough handling for unknown intents.

In [ ]:
# 🧪 Exercise: Add a code skill

@tool
def write_code(language: str, task: str) -> str:
    """Write code for a given task.
    
    Args:
        language: Programming language (python, javascript, etc.)
        task: What the code should do
    
    Returns:
        Code snippet
    """
    if language.lower() == "python":
        return f"# Python code for: {task}\ndef {task.replace(' ', '_') }():\n    pass"
    return f"// {language} code for: {task}"

---

## ❓ FAQ

**Q: Should I use rule-based or LLM-based routing?**
A: LLM-based is more flexible, rule-based is more predictable.

**Q: Can a single query need multiple skills?**
A: Yes! You can parallelize or chain skills.

**Q: What happens if routing is wrong?**
A: The response will be wrong - always test routing!

**Q: Can users help routing with intent?**
A: Absolutely! Explicit intent improves accuracy.

---

## ✅ Summary

You learned:

1. **Routing** = directing to the right handler
2. **Skill-based routing** = multiple specialized components
3. **Intent classification** = determining user goal
4. **Multi-skill agents** = scalable agent architecture

**Key insight**: Good routing is essential for agent evals - wrong routing = wrong answers!

---

## 🔗 Next Steps

Next: **[04_memory_and_state.ipynb](04_memory_and_state.ipynb)** - Learn how agents remember!